In [2]:
import pandas as pd
import xml.etree.ElementTree as ET
import torch
from BERT_Inference_Without_Finetune import get_embeddings_batch as get_embeddings_transformers
from gguf_utils import get_embeddings_batch as get_embeddings_gguf
import torch.nn.functional as F


/home/shapirma/.conda/envs/mind-the-gap/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from dataclasses import dataclass
from typing import List, Protocol, Dict
from itertools import chain
import numpy as np

class EmbeddingConfig(Protocol):
    """Common interface all embedding configs must satisfy"""
    texts: List[str]
    batch_size: int
    
    @property
    def embedding_key(self) -> str:
        """Unique key name for this embedding type"""
        ...
    
    def get_embeddings(self) -> List:
        ...

@dataclass
class TransformerConfig:
    model_name: str
    batch_size: int
    texts: List[str]
    
    @property
    def embedding_key(self) -> str:
        """Generate key from model name, e.g. 'bert_base_uncased_embedding'"""
        return f"{self.model_name.replace('-', '_')}_embedding"
    
    def get_embeddings(self):
        # Signature: get_embeddings_batch(texts, model_name, batch_size)
        return get_embeddings_transformers(self.texts, self.model_name, self.batch_size)

@dataclass
class GGUFConfig:
    repo_id: str
    filename: str
    batch_size: int
    texts: List[str]
    n_ctx: int = 2048
    n_threads: int = 8
    
    def _get_model_path(self) -> str:
        """Download GGUF model from HuggingFace Hub and return local path."""
        from huggingface_hub import hf_hub_download
        return hf_hub_download(repo_id=self.repo_id, filename=self.filename)
    
    @property
    def embedding_key(self) -> str:
        """Generate key from filename, e.g. 'OLMo_1B_Base_shakespeare_embedding'"""
        base_name = self.filename.split('.')[0]  # Remove extension
        return f"{base_name.replace('-', '_')}_embedding"
    
    def get_embeddings(self):
        # Signature: get_embeddings_batch(model_path, texts, batch_size, n_ctx, n_threads)
        model_path = self._get_model_path()
        return get_embeddings_gguf(
            model_path, self.texts, self.batch_size, self.n_ctx, self.n_threads
        )

In [4]:
# XML namespaces for TEI format
namespaces = {
    'tei': 'http://www.tei-c.org/ns/1.0',
    'xml': 'http://www.w3.org/XML/1998/namespace',
}

In [5]:
def find_tag_occurrences(root, tag_name):
    """Find all elements with a specific tag name in TEI namespace."""
    return root.findall(f'.//tei:{tag_name}', namespaces)


In [6]:
def get_characters_speeches(root):
    """
    Returns list(list(dict))
    Each list(dict) contains the speeches of the characters in the scene
    Each dict contains the speaker's name and the speech text
    The list(list(dict)) contains all the scenes in the play
    """

    characters_speeches = []

    scenes = find_tag_occurrences(root, 'div2')
    
    # Collect all speeches first
    for scene in scenes:
        scene_speeches = []
        for speech in scene.findall('tei:sp', namespaces):
            who = speech.get('who') # Remove the # symbol from the speaker id
            speech_info = {
                # TODO: Handle multiple speakers in a single speech
                'speaker': who.split()[0][1:] if who is not None else '[UNKNOWN]', # Remove the # symbol from the speaker id
                'text': "",
            }
            # Should be only one tei:ab in each sp tag
            ab_element = speech.find('tei:ab', namespaces)
            if ab_element is not None:
                # milestone_correspondence = speech.get('corresp').split(' ')
                for word in ab_element:
                    speech_info['text'] += word.text if word.text is not None else ""
            scene_speeches.append(speech_info)
        characters_speeches.append(scene_speeches)
        
    return characters_speeches # list(list(dict))



In [7]:
def flatten_speeches(characters_speeches: List[List[dict]]):
    """Flatten nested scene/speech structure into single iterator."""
    return chain.from_iterable(characters_speeches)


def to_numpy(embeddings) -> np.ndarray:
    """Convert embeddings to numpy array, handling torch tensors (CPU/GPU)."""
    if isinstance(embeddings, np.ndarray):
        return embeddings
    if hasattr(embeddings, 'cpu'):  # torch tensor
        return embeddings.detach().cpu().numpy()
    # List of tensors or arrays
    if isinstance(embeddings, list) and len(embeddings) > 0:
        if hasattr(embeddings[0], 'cpu'):  # list of torch tensors
            return np.stack([e.detach().cpu().numpy() for e in embeddings])
    return np.array(embeddings)


def compute_all_embeddings(
    configs: List[EmbeddingConfig]
) -> Dict[str, np.ndarray]:
    """
    Compute embeddings for all configs and return as dict.
    
    Args:
        configs: List of embedding configs (Transformer, GGUF, etc.)
    
    Returns:
        Dict mapping embedding_key -> numpy array of shape (num_speeches, embedding_dim)
    """
    embeddings = {}
    for config in configs:
        raw_embeddings = config.get_embeddings()
        embeddings[config.embedding_key] = to_numpy(raw_embeddings)
    return embeddings


def build_speeches_dataframe(characters_speeches: List[List[dict]]) -> pd.DataFrame:
    """
    Convert nested speech structure to flat DataFrame with metadata.
    
    Args:
        characters_speeches: Nested list of speech dicts by scene
    
    Returns:
        DataFrame with columns: scene_idx, speech_idx, speaker, text, text_length
    """
    records = []
    for scene_idx, scene in enumerate(characters_speeches):
        for speech_idx, speech in enumerate(scene):
            records.append({
                'scene_idx': scene_idx,
                'speech_idx': speech_idx,
                'speaker': speech['speaker'],
                'text': speech['text'],
                'text_length': len(speech['text'])
            })
    return pd.DataFrame(records)

In [8]:
def get_unique_characters(df: pd.DataFrame) -> List[str]:
    """Get list of unique characters from DataFrame."""
    return df['speaker'].unique().tolist()


def build_character_relations(
    df: pd.DataFrame, 
    embeddings: Dict[str, np.ndarray]
) -> pd.DataFrame:
    """
    Build a unified DataFrame with co-occurrences and cosine similarities for all character pairs.
    
    For each character pair (A, B):
    - Sum all embeddings from A's speeches in conversations with B
    - Sum all embeddings from B's speeches in conversations with A  
    - Compute cosine similarity between these summed embedding vectors
    
    Args:
        df: DataFrame with speech metadata (must have 'scene_idx', 'speaker' columns)
        embeddings: Dict mapping embedding_key -> numpy array aligned with df index
    
    Returns:
        DataFrame with MultiIndex (character1, character2) and columns:
        - co_occurrence: int count of consecutive speeches
        - {embedding_key}_similarity: float cosine similarity of summed embeddings
    """
    characters = get_unique_characters(df)
    embedding_dim = {key: emb.shape[1] for key, emb in embeddings.items()}
    
    # Create all character pairs (excluding self-pairs)
    pairs = [(c1, c2) for c1 in characters for c2 in characters if c1 != c2]
    index = pd.MultiIndex.from_tuples(pairs, names=['character1', 'character2'])
    
    # Initialize result DataFrame
    columns = ['co_occurrence'] + [f"{key}_similarity" for key in embeddings.keys()]
    result = pd.DataFrame(0.0, index=index, columns=columns)
    result['co_occurrence'] = result['co_occurrence'].astype(int)
    
    # Track summed embeddings for each character in each pair
    # emb_sums[key][(c1, c2)] = sum of c1's embeddings when talking with c2
    emb_sums = {
        key: {pair: np.zeros(embedding_dim[key]) for pair in pairs}
        for key in embeddings.keys()
    }
    
    # Process each scene
    for scene_idx, scene_df in df.groupby('scene_idx'):
        indices = scene_df.index.tolist()
        speakers = scene_df['speaker'].tolist()
        
        for i in range(len(indices) - 1):
            speaker1, speaker2 = speakers[i], speakers[i + 1]
            
            if speaker1 != speaker2:
                idx1, idx2 = indices[i], indices[i + 1]
                
                # Update co-occurrence (both directions for symmetry)
                result.loc[(speaker1, speaker2), 'co_occurrence'] += 1
                result.loc[(speaker2, speaker1), 'co_occurrence'] += 1
                
                # Sum embeddings for each character in the pair
                for key, emb_array in embeddings.items():
                    # speaker1's embedding goes to (speaker1, speaker2) pair
                    emb_sums[key][(speaker1, speaker2)] += emb_array[idx1]
                    # speaker2's embedding goes to (speaker2, speaker1) pair
                    emb_sums[key][(speaker2, speaker1)] += emb_array[idx2]
    
    # Calculate cosine similarity from summed embeddings
    for key in embeddings.keys():
        col_name = f"{key}_similarity"
        for c1, c2 in pairs:
            sum_c1 = emb_sums[key][(c1, c2)]  # c1's summed embeddings when talking to c2
            sum_c2 = emb_sums[key][(c2, c1)]  # c2's summed embeddings when talking to c1
            
            norm_c1 = np.linalg.norm(sum_c1)
            norm_c2 = np.linalg.norm(sum_c2)
            
            if norm_c1 > 0 and norm_c2 > 0:
                similarity = np.dot(sum_c1, sum_c2) / (norm_c1 * norm_c2)
                result.loc[(c1, c2), col_name] = similarity
            else:
                result.loc[(c1, c2), col_name] = 0.0
    
    return result


In [9]:
# Load and parse XML
hamlet_path = '../Data/XML/hamlet_XML_FolgerShakespeare/Ham.xml'
with open(hamlet_path, 'r') as file:
    hamlet_xml = ET.fromstring(file.read())

# Extract speeches from XML
characters_speeches = get_characters_speeches(hamlet_xml)

# Build metadata DataFrame (no embeddings - just text and metadata)
df = build_speeches_dataframe(characters_speeches)

# Get texts for embedding computation
speeches_texts = df['text'].tolist()

# Define embedding configs
configs = [
    TransformerConfig(
        model_name="bert-base-uncased", 
        batch_size=16, 
        texts=speeches_texts
    ),
    GGUFConfig(
        repo_id="mradermacher/OLMo-1B-Base-shakespeare-GGUF", 
        filename="OLMo-1B-Base-shakespeare.IQ3_M.gguf",
        batch_size=16, 
        texts=speeches_texts
    )
]

# Compute all embeddings - stored separately from DataFrame
# embeddings dict: {"bert_base_uncased_embedding": array(N, 768), "OLMo_1B_Base_shakespeare_embedding": array(N, dim)}
embeddings = compute_all_embeddings(configs)


2026-01-04 15:27:53.755125: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
llama_context: n_ctx_per_seq (2048) < n_ctx_train (4096) -- the full capacity of the model will not be utilized


In [20]:
df['text'].tolist()

['Who’s there?',
 'Nay, answer me. Stand and unfold yourself.',
 'Long live the King!',
 'Barnardo?',
 'He.',
 'You come most carefully upon your hour.',
 '’Tis now struck twelve. Get thee to bed, Francisco.',
 'For this relief much thanks. ’Tis bitter cold,And I am sick at heart.',
 'Have you had quiet guard?',
 'Not a mouse stirring.',
 'Well, good night.If you do meet Horatio and Marcellus,The rivals of my watch, bid them make haste.',
 'I think I hear them.—Stand ho! Who is there?',
 'Friends to this ground.',
 'And liegemen to the Dane.',
 'Give you good night.',
 'O farewell, honest soldier. Who hath relievedyou?',
 'Barnardo hath my place. Give you good night.',
 'Holla, Barnardo.',
 'Say, what, is Horatio there?',
 'A piece of him.',
 'Welcome, Horatio.—Welcome, good Marcellus.',
 'What, has this thing appeared again tonight?',
 'I have seen nothing.',
 'Horatio says ’tis but our fantasyAnd will not let belief take hold of himTouching this dreaded sight twice seen of us.Therefo

In [10]:
# Build unified character relations DataFrame
# Contains co-occurrences AND similarities in one DataFrame with MultiIndex
character_relations = build_character_relations(df, embeddings)

print(f"Character relations shape: {character_relations.shape}")
print(f"Columns: {character_relations.columns.tolist()}")
print(f"Index levels: {character_relations.index.names}")


Character relations shape: (1482, 3)
Columns: ['co_occurrence', 'bert_base_uncased_embedding_similarity', 'OLMo_1B_Base_shakespeare_embedding_similarity']
Index levels: ['character1', 'character2']


In [11]:
# View the unified character relations DataFrame
character_relations.head(20)


co_occurrence  \
character1   character2                            
Barnardo_Ham Francisco_Ham                    11   
             Horatio_Ham                      18   
             Marcellus_Ham                    10   
             Claudius_Ham                      0   
             Cornelius_Ham                     0   
             Laertes_Ham                       0   
             Polonius_Ham                      0   
             Hamlet_Ham                        0   
             Gertrude_Ham                      0   
             Ophelia_Ham                       0   
             Ghost_Ham                         0   
             Reynaldo_Ham                      0   
             Rosencrantz_Ham                   0   
             Guildenstern_Ham                  0   
             Voltemand_Ham                     0   
             PLAYERS.1_Ham                     0   
             PLAYERS.0.1_Ham                   0   
             PLAYERS.Prologue_Ham              0   
             PLAYERS.King_Ham                  0   
             PLAYERS.Queen_Ham                 0   

                                   bert_base_uncased_embedding_similarity  \
character1   character2                                                     
Barnardo_Ham Francisco_Ham                                       0.905546   
             Horatio_Ham                                         0.958068   
             Marcellus_Ham                                       0.905651   
             Claudius_Ham                                        0.000000   
             Cornelius_Ham                                       0.000000   
             Laertes_Ham                                         0.000000   
             Polonius_Ham                                        0.000000   
             Hamlet_Ham                                          0.000000   
             Gertrude_Ham                                        0.000000   
             Ophelia_Ham                                         0.000000   
             Ghost_Ham                                           0.000000   
             Reynaldo_Ham                                        0.000000   
             Rosencrantz_Ham                                     0.000000   
             Guildenstern_Ham                                    0.000000   
             Voltemand_Ham                                       0.000000   
             PLAYERS.1_Ham                                       0.000000   
             PLAYERS.0.1_Ham                                     0.000000   
             PLAYERS.Prologue_Ham                                0.000000   
             PLAYERS.King_Ham                                    0.000000   
             PLAYERS.Queen_Ham                                   0.000000   

                                   OLMo_1B_Base_shakespeare_embedding_similarity  
character1   character2                                                           
Barnardo_Ham Francisco_Ham                                              0.673902  
             Horatio_Ham                                                0.885760  
             Marcellus_Ham                                              0.797901  
             Claudius_Ham                                               0.000000  
             Cornelius_Ham                                              0.000000  
             Laertes_Ham                                                0.000000  
             Polonius_Ham                                               0.000000  
             Hamlet_Ham                                                 0.000000  
             Gertrude_Ham                                               0.000000  
             Ophelia_Ham                                                0.000000  
             Ghost_Ham                                                  0.000000  
             Reynaldo_Ham                                               0.000000  
             Rosencrantz_Ham     

In [12]:
# Example queries on the unified DataFrame

# Get a specific character pair's stats
print("Hamlet <-> Claudius relationship:")
print(character_relations.loc[('Hamlet_Ham', 'Polonius_Ham')])

print("\n" + "="*50 + "\n")

# Get all relationships for a specific character
print("All of Hamlet's relationships (sorted by co-occurrence):")
hamlet_relations = character_relations.xs('Hamlet_Ham', level='character1')
hamlet_relations.sort_values('co_occurrence', ascending=False).head(10)


Hamlet <-> Claudius relationship:
co_occurrence                                    73.000000
bert_base_uncased_embedding_similarity            0.939318
OLMo_1B_Base_shakespeare_embedding_similarity     0.917994
Name: (Hamlet_Ham, Polonius_Ham), dtype: float64


All of Hamlet's relationships (sorted by co-occurrence):


,co_occurrence,bert_base_uncased_embedding_similarity,OLMo_1B_Base_shakespeare_embedding_similarity
character2,,,
Horatio_Ham,158,0.938514,0.892979
Polonius_Ham,73,0.939318,0.917994
Rosencrantz_Ham,70,0.954417,0.889340
Gertrude_Ham,66,0.971669,0.914074
Ophelia_Ham,54,0.899185,0.848808
Claudius_Ham,51,0.961573,0.889400
Guildenstern_Ham,43,0.969232,0.885137
Gravedigger_Ham,41,0.916478,0.862135
Osric_Ham,39,0.968253,0.915468


In [13]:
# Convert to matrix view if needed (for heatmaps, etc.)
def to_matrix(character_relations: pd.DataFrame, column: str) -> pd.DataFrame:
    """Pivot the MultiIndex DataFrame back to character x character matrix."""
    return character_relations[column].unstack(level='character2')

# Co-occurrence matrix view
co_oc_matrix = to_matrix(character_relations, 'co_occurrence')
print("Co-occurrence matrix (first 6 characters):")
co_oc_matrix.iloc[:6, :6]

Co-occurrence matrix (first 6 characters):


character2,AMBASSADORS_Ham,ATTENDANTS.0.1_Ham,ATTENDANTS.1_Ham,ATTENDANTS.2_Ham,ATTENDANTS.GUARDS_Ham,ATTENDANTS_Ham
character1,,,,,,
AMBASSADORS_Ham,NaN,0.0,0.0,0.0,0.0,0.0
ATTENDANTS.0.1_Ham,0.0,NaN,0.0,0.0,0.0,0.0
ATTENDANTS.1_Ham,0.0,0.0,NaN,0.0,0.0,0.0
ATTENDANTS.2_Ham,0.0,0.0,0.0,NaN,0.0,0.0
ATTENDANTS.GUARDS_Ham,0.0,0.0,0.0,0.0,NaN,0.0
ATTENDANTS_Ham,0.0,0.0,0.0,0.0,0.0,NaN


In [14]:
# Example: How to use DataFrame + embeddings together

# View the metadata
print(f"DataFrame shape: {df.shape}")
print(f"Embeddings keys: {list(embeddings.keys())}")
for key, emb in embeddings.items():
    print(f"  {key}: shape {emb.shape}")

df.head()

DataFrame shape: (1138, 5)
Embeddings keys: ['bert_base_uncased_embedding', 'OLMo_1B_Base_shakespeare_embedding']
  bert_base_uncased_embedding: shape (1138, 768)
  OLMo_1B_Base_shakespeare_embedding: shape (1138, 2048)


,scene_idx,speech_idx,speaker,text,text_length
0,0,0,Barnardo_Ham,Who’s there?,12
1,0,1,Francisco_Ham,"Nay, answer me. Stand and unfold yourself.",42
2,0,2,Barnardo_Ham,Long live the King!,19
3,0,3,Francisco_Ham,Barnardo?,9
4,0,4,Barnardo_Ham,He.,3


In [15]:
# Example: Filter by speaker and get their embeddings

def get_speaker_embeddings(df: pd.DataFrame, embeddings: Dict[str, np.ndarray], 
                           speaker: str, embedding_key: str) -> np.ndarray:
    """Get embeddings for a specific speaker."""
    speaker_idxs = df[df['speaker'] == speaker].index.values
    return embeddings[embedding_key][speaker_idxs]


# Get unique speakers
print("Unique speakers:", df['speaker'].unique()[:10], "...")

# Get Hamlet's speeches and embeddings
hamlet_df = df[df['speaker'] == 'Hamlet_Ham']
print(f"\nHamlet has {len(hamlet_df)} speeches")

# Get Hamlet's BERT embeddings (using aligned indices)
bert_key = configs[0].embedding_key
hamlet_bert_embeddings = embeddings[bert_key][hamlet_df.index.values]
print(f"Hamlet's BERT embeddings shape: {hamlet_bert_embeddings.shape}")


Unique speakers: ['Barnardo_Ham' 'Francisco_Ham' 'Horatio_Ham' 'Marcellus_Ham'
 'Claudius_Ham' 'Cornelius_Ham' 'Laertes_Ham' 'Polonius_Ham' 'Hamlet_Ham'
 'Gertrude_Ham'] ...

Hamlet has 358 speeches
Hamlet's BERT embeddings shape: (358, 768)


In [16]:
# Example: Export to parquet with embeddings (only when needed for storage/sharing)

def export_with_embeddings(df: pd.DataFrame, embeddings: Dict[str, np.ndarray], 
                           output_path: str) -> None:
    """Export DataFrame with embeddings to parquet file."""
    df_export = df.copy()
    for key, emb in embeddings.items():
        df_export[key] = list(emb)  # list() keeps as arrays in cells
    df_export.to_parquet(output_path)
    print(f"Exported to {output_path}")


# Uncomment to export:
# export_with_embeddings(df, embeddings, 'hamlet_speeches_with_embeddings.parquet')
